# OCR Medicine Matcher Benchmark

This notebook benchmarks the real fuzzy matching logic used by the SahiDawa ML service.

It imports `find_matches()` from `apps/ml/services/matcher.py` and evaluates it on noisy OCR-style medicine names.


In [ ]:
!pip install thefuzz pydantic pandas matplotlib

In [ ]:
!git clone https://github.com/RatLoopz/sahidawa-india.git

In [ ]:
import sys
sys.path.append("sahidawa-india/apps/ml")

import pandas as pd
import matplotlib.pyplot as plt

from services.matcher import find_matches

In [ ]:
medicine_names = [
    "Paracetamol",
    "Azithromycin",
    "Amoxicillin",
    "Cetirizine",
    "Ibuprofen",
    "Metformin",
    "Aspirin",
    "Omeprazole",
    "Dolo 650",
    "Crocin"
]

ocr_samples = [
    ("Paracetamo1", "Paracetamol"),
    ("Azithromyc1n", "Azithromycin"),
    ("Amoxcillin", "Amoxicillin"),
    ("Cetirizin", "Cetirizine"),
    ("Ibuprofenn", "Ibuprofen"),
    ("Metfomin", "Metformin"),
    ("Asprin", "Aspirin"),
    ("Omeprazol", "Omeprazole"),
    ("Dolo650", "Dolo 650"),
    ("Crocim", "Crocin")
]

In [ ]:
results = []

for noisy_text, expected in ocr_samples:
    matches = find_matches(noisy_text, medicine_names, limit=3)

    top_predictions = [match.name for match in matches]
    top_scores = [match.score for match in matches]

    results.append({
        "OCR Text": noisy_text,
        "Expected": expected,
        "Top Match": top_predictions[0] if top_predictions else None,
        "Top Score": top_scores[0] if top_scores else None,
        "Top-1 Correct": top_predictions[0] == expected if top_predictions else False,
        "Top-3 Correct": expected in top_predictions
    })

df = pd.DataFrame(results)
df

In [ ]:
top1_accuracy = df["Top-1 Correct"].mean() * 100
top3_accuracy = df["Top-3 Correct"].mean() * 100
average_score = df["Top Score"].mean()

print(f"Top-1 Accuracy: {top1_accuracy:.2f}%")
print(f"Top-3 Accuracy: {top3_accuracy:.2f}%")
print(f"Average Confidence Score: {average_score:.2f}")

In [ ]:
metrics = pd.DataFrame({
    "Metric": ["Top-1 Accuracy", "Top-3 Accuracy", "Average Confidence Score"],
    "Value": [top1_accuracy, top3_accuracy, average_score]
})

metrics

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(metrics["Metric"], metrics["Value"])
plt.title("OCR Medicine Matching Benchmark")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.show()